In [ ]:
# imports and configuration

from transformers import AutoTokenizer, AutoModelForMaskedLM, GPTQConfig
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import sys
import copy

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

project_root = Path("/home/lecacca/VSCode Environment/BERT-Testing")
helper_dir = project_root / "scripts" / "quantize_sublayer"

if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

import quantize_experiments_gptq_implementation as manual_helpers
import quantize_experiments_gptq as gptq_helpers

model_name = "/home/lecacca/VSCode Environment/BERT-Testing/bert_uncased_L-2_H-128_A-2"

texts_path = Path(
    "/home/lecacca/VSCode Environment/BERT-Testing/visualization/scatter_plot_gptq/masked_texts.txt"
)

default_out_dir = Path(
    "/home/lecacca/VSCode Environment/BERT-Testing/visualization/scatter_plot_gptq/results"
)

out_dir = Path(os.environ.get("RESULTS_DIR", default_out_dir))
out_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", out_dir)

block_index = 0

sublayers = [
    "attention.self.query",
    "attention.self.key",
    "attention.self.value",
    "attention.output.dense",
    "intermediate.dense",
    "output.dense",
]

In [ ]:
# load masked texts

def load_masked_texts(path):
    texts = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            texts.append(line)

    return texts


masked_texts = load_masked_texts(texts_path)

print(f"Loaded {len(masked_texts)} masked texts")
for i, text in enumerate(masked_texts[:5]):
    print(f"{i}: {text}")

In [ ]:
# notebook helpers

def get_mask_logits(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt")
    inputs = manual_helpers.move_inputs_to_model_device(inputs, model)

    mask_positions = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=False)

    if mask_positions.numel() == 0:
        raise ValueError(f"No [MASK] token found in text: {text}")

    mask_index = mask_positions[0, 1].item()

    with torch.inference_mode():
        outputs = model(**inputs)

    logits = outputs.logits
    mask_logits = logits[0, mask_index, :].detach().float().cpu()

    return mask_logits, inputs


def get_top1_token_from_logits(logits, tokenizer):
    top_idx = torch.argmax(logits).item()
    return tokenizer.convert_ids_to_tokens([top_idx])[0]


def get_topk_tokens_from_logits(logits, tokenizer, k=5):
    top_values, top_indices = torch.topk(logits, k)

    tokens = tokenizer.convert_ids_to_tokens(
        top_indices.detach().cpu().tolist()
    )
    scores = top_values.detach().cpu().tolist()

    return tokens, scores

In [ ]:
# GPTQ helper for one-sublayer quantization

def load_gptq_model_for_one_sublayer(model_name, tokenizer, sub_layer, block_index=0):
    calibration_texts = masked_texts[: min(len(masked_texts), 16)]

    gptq_config = GPTQConfig(
        bits=8,
        dataset=calibration_texts,
        tokenizer=tokenizer,
        group_size=128,
        desc_act=False,
        sym=False,
        block_name_to_quantize="bert.encoder.layer",
        modules_in_block_to_quantize=[[sub_layer]],
    )

    model = AutoModelForMaskedLM.from_pretrained(
        model_name,
        quantization_config=gptq_config,
        device_map="auto",
    )
    model.eval()

    return model

In [ ]:
# precompute FP32 baseline logits

tokenizer = AutoTokenizer.from_pretrained(model_name)

baseline_model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
baseline_model.eval()

baseline_logits_by_text = {}
baseline_top1_by_text = {}

for text_index, text in enumerate(masked_texts):
    logits, _ = get_mask_logits(baseline_model, tokenizer, text)

    baseline_logits_by_text[text_index] = logits
    baseline_top1_by_text[text_index] = get_top1_token_from_logits(logits, tokenizer)

print("Finished baseline inference.")
print("Example baseline top-1 predictions:")

for i in range(min(5, len(masked_texts))):
    print(i, masked_texts[i], "->", baseline_top1_by_text[i])

In [ ]:
def capture_module_io(model, module, inputs):
    saved = {}
    inputs = manual_helpers.move_inputs_to_model_device(inputs, model)

    def hook_fn(module, input, output):
        saved["input"] = input[0].detach()
        saved["output"] = output.detach()

    handle = module.register_forward_hook(hook_fn)

    with torch.no_grad():
        _ = model(**inputs)

    handle.remove()
    return saved


def recompute_linear_output(input_tensor, layer):
    weight = layer.weight.detach().to(input_tensor.device)
    bias = layer.bias.detach().to(input_tensor.device) if layer.bias is not None else None

    output = torch.matmul(input_tensor, weight.T)
    if bias is not None:
        output = output + bias

    return output

In [ ]:
# run manual-vs-GPTQ quantization experiments

results = []

for sub_layer in sublayers:
    print(f"\n=== Sublayer: block {block_index}.{sub_layer} ===")

    # ------------------------------------------------------------
    # Fresh model for manual quantization
    # ------------------------------------------------------------
    manual_model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
    manual_model.eval()

    manual_helpers.apply_manual_gptq_to_one_sublayer(
        manual_model,
        tokenizer=tokenizer,
        calibration_texts=masked_texts[: min(len(masked_texts), 16)],
        block_index=block_index,
        sub_layer=sub_layer,
        num_bits=8,
        group_size=128,
        sym=False,
        damping_percent=0.01,
        verbose=False,
    )

    manual_layer = manual_helpers.get_module_by_name(
        manual_model.bert.encoder.layer[block_index],
        sub_layer,
    )

    # ------------------------------------------------------------
    # Fresh model for GPTQ quantization
    # ------------------------------------------------------------
    gptq_model = load_gptq_model_for_one_sublayer(
        model_name,
        tokenizer,
        sub_layer=sub_layer,
        block_index=block_index,
    )
    gptq_model.eval()

    gptq_layer = manual_helpers.get_module_by_name(
        gptq_model.bert.encoder.layer[block_index],
        sub_layer,
    )

    # ------------------------------------------------------------
    # Baseline layer
    # ------------------------------------------------------------
    baseline_layer = manual_helpers.get_module_by_name(
        baseline_model.bert.encoder.layer[block_index],
        sub_layer,
    )

    # ------------------------------------------------------------
    # Evaluate all masked texts
    # ------------------------------------------------------------
    for text_index, text in enumerate(masked_texts):
        baseline_logits = baseline_logits_by_text[text_index]

        # inputs for this text
        _, inputs = get_mask_logits(baseline_model, tokenizer, text)

        # capture sublayer I/O for all three models
        baseline_capture = capture_module_io(
            baseline_model,
            baseline_layer,
            inputs,
        )

        manual_logits, _ = get_mask_logits(manual_model, tokenizer, text)
        manual_capture = capture_module_io(
            manual_model,
            manual_layer,
            inputs,
        )

        gptq_logits, _ = get_mask_logits(gptq_model, tokenizer, text)
        gptq_capture = capture_module_io(
            gptq_model,
            gptq_layer,
            inputs,
        )

        # baseline expected outputs on the exact inputs seen by each path
        baseline_expected_manual = recompute_linear_output(
            manual_capture["input"],
            baseline_layer,
        )

        baseline_expected_gptq = recompute_linear_output(
            gptq_capture["input"],
            baseline_layer,
        )

        # local activation drift
        manual_activation = manual_helpers.compare_activation_drift(
            baseline_expected_manual,
            manual_capture["output"],
            label=f"manual activation drift {sub_layer}",
        )

        gptq_activation = manual_helpers.compare_activation_drift(
            baseline_expected_gptq,
            gptq_capture["output"],
            label=f"gptq activation drift {sub_layer}",
        )

        # final logit drift
        manual_abs_diff = (manual_logits - baseline_logits).abs()
        gptq_abs_diff = (gptq_logits - baseline_logits).abs()

        manual_mean_abs_diff = manual_abs_diff.mean().item()
        manual_max_abs_diff = manual_abs_diff.max().item()

        gptq_mean_abs_diff = gptq_abs_diff.mean().item()
        gptq_max_abs_diff = gptq_abs_diff.max().item()

        manual_top1 = get_top1_token_from_logits(manual_logits, tokenizer)
        gptq_top1 = get_top1_token_from_logits(gptq_logits, tokenizer)
        baseline_top1 = baseline_top1_by_text[text_index]

        results.append({
            "block_index": block_index,
            "sub_layer": sub_layer,
            "text_index": text_index,
            "text": text,

            "baseline_top1": baseline_top1,
            "manual_top1": manual_top1,
            "gptq_top1": gptq_top1,

            "manual_top1_match": manual_top1 == baseline_top1,
            "gptq_top1_match": gptq_top1 == baseline_top1,

            "manual_mean_abs_logit_diff": manual_mean_abs_diff,
            "manual_max_abs_logit_diff": manual_max_abs_diff,

            "gptq_mean_abs_logit_diff": gptq_mean_abs_diff,
            "gptq_max_abs_logit_diff": gptq_max_abs_diff,

            "manual_activation_rel_l2": manual_activation["rel_l2"],
            "manual_activation_cosine": manual_activation["cosine"],

            "gptq_activation_rel_l2": gptq_activation["rel_l2"],
            "gptq_activation_cosine": gptq_activation["cosine"],
        })

    del manual_model
    del gptq_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df = pd.DataFrame(results)

print("Finished experiments.")
display(df.head())

out_csv = out_dir / "manual_vs_gptq_results.csv"
df.to_csv(out_csv, index=False)

print("Saved:", out_csv)

In [ ]:
# scatter plot: manual vs GPTQ activation drift

plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=df,
    x="manual_activation_rel_l2",
    y="gptq_activation_rel_l2",
    hue="sub_layer",
    style="sub_layer",
    s=80,
    alpha=0.85,
)

max_val = max(
    df["manual_activation_rel_l2"].max(),
    df["gptq_activation_rel_l2"].max(),
)

plt.plot([0, max_val], [0, max_val], "k--", linewidth=1, label="y = x")

plt.xlabel("Manual quantization relative L2 activation drift")
plt.ylabel("GPTQ relative L2 activation drift")

plt.title(
    f"Manual vs GPTQ Quantization\n"
    f"Block {block_index}, local activation drift"
)

plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

fig_path = out_dir / "scatter_manual_vs_gptq_activation_rel_l2.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")

print("Saved:", fig_path)

plt.show()

In [ ]:
# scatter plot: manual vs GPTQ mean absolute logit difference

plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=df,
    x="manual_mean_abs_logit_diff",
    y="gptq_mean_abs_logit_diff",
    hue="sub_layer",
    style="sub_layer",
    s=80,
    alpha=0.85,
)

max_val = max(
    df["manual_mean_abs_logit_diff"].max(),
    df["gptq_mean_abs_logit_diff"].max(),
)

plt.plot([0, max_val], [0, max_val], "k--", linewidth=1, label="y = x")

plt.xlabel("Manual quantization mean absolute logit diff")
plt.ylabel("GPTQ mean absolute logit diff")

plt.title(
    f"Manual vs GPTQ Quantization\n"
    f"Block {block_index}, mean absolute [MASK] logit drift"
)

plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

fig_path = out_dir / "scatter_manual_vs_gptq_mean_abs_logit_diff.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")

print("Saved:", fig_path)

plt.show()